In [0]:
select 
p.Categoria_Producto
,v.Region
,v.Canal_Venta
,count(distinct v.ID_Venta) as Num_pedidos_Unicos
,sum(v.Total_Venta) as Ventas_Totales

from catalogo.ventas.raw_ventas v
join catalogo.ventas.productos_dim p on v.ID_Producto = p.ID_Producto
where v.Fecha_Venta >= '2023-01-01' and v.Fecha_Venta <= '2023-03-31'
group by p.Categoria_Producto
,v.Region
,v.Canal_Venta
order by Ventas_Totales desc;
-----------------------------------------------------------------------------------uso explain
explain formatted
SELECT * FROM catalogo.ventas.raw_ventas
WHERE Region = 'Norte' and Total_Venta > 500;





-------------------------------------------------------------------------------------------uso del Z order 

DESCRIBE DETAIL catalogo.ventas.raw_ventas;


OPTIMIZE catalogo.ventas.raw_ventas
ZORDER BY (Region,Categoria_Producto);



--------------------------------------------------------------------------------------------liquid cluster


CREATE OR REPLACE TABLE catalogo.ventas.ventas_liquid
CLUSTER BY (ID_Producto, Region)
AS
SELECT * FROM catalogo.ventas.raw_ventas;


SELECT * FROM catalogo.ventas.ventas_liquid;

-- Mantenimiento incremental

optimize catalogo.ventas.ventas_liquid;

-- Cambiando la Estrategia sin reescribir datos historicos

ALTER TABLE catalogo.ventas.ventas_liquid
CLUSTER BY (ID_Producto, Region, Fecha_Venta);


------------------------

sELECT count(*), avg(Total_Venta) FROM catalogo.ventas.fact_ventas GROUP BY Region;


----------------------------------------------------------comandos acceso a tablas o volumness


GRANT USE CATALOG ON catalogo TO `account users`;

GRANT USE SCHEMA ON SCHEMA catalogo.ventas TO `account users`;



GRANT SELECT ON TABLE catalogo.ventas.raw_ventas TO `account users`;

SHOW GRANTS ON TABLE catalogo.ventas.raw_ventas;



------------------------------error 

GRANT ALL PRIVILEGES ON TABLE catalogo.ventas.raw_ventas TO `account users`;



-- Error
REVOKE ALL PRIVILEGES ON  TABLE catalogo.ventas.raw_ventas FROM `account users`;

--restaurando solo el select 

GRANT SELECT ON TABLE catalogo.ventas.raw_ventas TO `account users`;


--------------------------------------------------------------------------comentar columnas

COMMENT ON TABLE catalogo.ventas.ventas_liquid IS 'Tabla maestra de ventas optimizada con Liquid Clustering';
COMMENT ON COLUMN catalogo.ventas.ventas_liquid.ID_Venta IS 'Identificador unico de la transaccion';
COMMENT ON COLUMN catalogo.ventas.ventas_liquid.total_venta IS 'Valor total de la transaccion en Euros, impuestos incluidos';



ALTER TABLE catalogo.ventas.ventas_liquid
SET TAGS ('Clasificacion','Confidencial','Departamento','Ventas');


ALTER TABLE catalogo.ventas.ventas_liquid
ALTER COLUMN Total_Venta SET TAGS ('Clasificacion' = 'True');



----------------------------RLS row level security 

--1 sae crea la vista con RLS

CREATE OR REPLACE VIEW ventas_rls as 
SELECT * FROM catalogo.ventas.raw_ventas
WHERE 

is_member('admins') OR  (Region = 'Norte' AND current_user = 'friederiksvae@gmail.com');

--2 se le da permiso a la vista

GRANT SELECT ON VIEW ventas_rls TO `account users`;

SELECT * FROM ventas_rls ;--- prueba que podemos ver la view


CREATE OR REPLACE VIEW ventas_masked AS 
SELECT 

ID_Venta
,ID_Producto
,Region
,CASE WHEN is_member('ABS') THEN Total_Venta ELSE -999 END AS Total_Venta_Segura------------------METODO DE ENMASCARAMIENTO  PARA USUARIOS GRUPALES.


FROM catalogo.ventas.raw_ventas;


SELECT * FROM ventas_masked
